# Cross-Cultural Tourist Review Analysis: Data Overview

Domestic (Korean-language) vs foreign (English-language) reviews of attractions in Seoul and Busan.
All tables are read from `data/analysis/`; figures are the ones produced by the pipeline scripts.
Re-run the pipeline (`run_pipeline.py`) and then this notebook to refresh everything.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "corpus.csv").exists())
DATA = ROOT / "data" / "analysis"
FIGS = DATA / "figures"

PLACE_EN = {
    "경복궁": "Gyeongbokgung", "창덕궁": "Changdeokgung", "덕수궁": "Deoksugung",
    "북촌한옥마을": "Bukchon Village", "인사동길": "Insadong-gil", "낙산공원": "Naksan Park",
    "광장시장": "Gwangjang Market", "홍대거리": "Hongdae Street", "국립중앙박물관": "National Museum",
    "N서울타워": "N Seoul Tower", "롯데월드": "Lotte World",
    "광안리해수욕장": "Gwangalli Beach", "해운대해수욕장": "Haeundae Beach",
}
ORDER = [p for p in PLACE_EN.values()]

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)


def read(name):
    return pd.read_csv(DATA / f"{name}.csv", encoding="utf-8-sig")


def en(df, col="place"):
    df = df.copy()
    df[col] = df[col].map(PLACE_EN).fillna(df[col])
    return df


def fig(name, width=900):
    path = FIGS / name
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"*Figure `{name}` not found: run the pipeline script that creates it.*"))


print("Project root:", ROOT)

## 1. Corpus

In [ ]:
corpus = pd.read_csv(ROOT / "data" / "corpus.csv", encoding="utf-8-sig")
table = en(corpus).pivot_table(index="place", columns="language", values="text", aggfunc="count", fill_value=0)
table = table.reindex([p for p in ORDER if p in table.index])
table.loc["TOTAL"] = table.sum()
print(f"{len(corpus)} reviews  |  KO = {(corpus.language == 'ko').sum()}  |  EN = {(corpus.language == 'en').sum()}")
table

## 2. Sentiment (-1 to 1) and Mann-Whitney U (one-sided, KO > EN)

In [ ]:
mw = en(read("stats_mannwhitney"))
mw["gap_ko_en"] = mw["ko_mean"] - mw["en_mean"]
mw[["place", "ko_mean", "en_mean", "gap_ko_en", "U_stat", "p_value", "sig"]]

In [ ]:
fig("01_sentiment_bar.png")
fig("12_sentiment_scatter.png", 650)

## 3. LDA topics and choice of K

In [ ]:
for lang in ("ko", "en"):
    display(Markdown(f"**{lang.upper()} topics (K = 5)**"))
    display(read(f"lda_{lang}_topics")[["topic", "top_words"]])

In [ ]:
k = read("lda_k_selection")
display(k.pivot(index="k", columns="language", values="coherence_mean").round(3))
fig("14_lda_k_selection.png")

## 4. TF-IDF keywords

In [ ]:
display(read("tfidf_contrast"))
fig("06_tfidf_contrast.png")

## 5. Lexical diversity (type-token ratio)

In [ ]:
ttr = en(read("ttr_results")).pivot(index="place", columns="language", values="mean")
ttr["gap_ko_en"] = ttr["ko"] - ttr["en"]
display(ttr.reindex([p for p in ORDER if p in ttr.index]))
fig("05_ttr_bar.png")

## 6. Cross-lingual semantic similarity

In [ ]:
display(en(read("crosslingual_similarity")))
fig("08_crosslingual_sim.png")

## 7. Travel motivation (zero-shot)

In [ ]:
mot = read("motivation_summary")
overall = mot.groupby(["motivation", "language"])["count"].sum().unstack()
overall = (overall / overall.sum() * 100).round(1)
overall["gap_ko_en"] = overall["ko"] - overall["en"]
display(overall.sort_values("ko", ascending=False))
fig("07_motivations_stacked.png")

## 8. Aspect-based sentiment (ABSA, -1 to +1)

In [ ]:
absa = en(read("absa_summary"))
aspects = [c for c in absa.columns if c.endswith("_mean")]
display(absa[["place", "language", *aspects]])
fig("10_absa_radar.png", 700)

## 9. Hofstede-based hypothesis tests

In [ ]:
ht = read("hypothesis_tests")
display(ht[["hypothesis", "dimension", "label", "ko_presence_pct", "en_presence_pct",
            "presence_p_holm", "presence_result", "ko_mean", "en_mean", "p_holm", "result"]])

In [ ]:
display(Markdown("**Sensitivity: presence test without terms that failed manual validation**"))
display(read("hypothesis_tests_sensitivity")[[
    "hypothesis", "label", "excluded_terms",
    "ko_presence_pct_full", "en_presence_pct_full", "presence_p_holm_full",
    "ko_presence_pct_reduced", "en_presence_pct_reduced", "presence_p_holm_reduced", "robust"]])
fig("13_hypothesis_tests.png")

In [ ]:
prec_path = DATA / "dictionary_precision.csv"
if prec_path.exists():
    prec = pd.read_csv(prec_path, encoding="utf-8-sig")
    display(Markdown("**Dictionary precision (manual KWIC coding)**"))
    display(prec.groupby(["hypothesis", "language"])
                .apply(lambda g: g["n_valid"].sum() / g["n_reviewed"].sum())
                .unstack().round(3))
    display(prec[prec["action"] == "EXCLUDE"])

## 10. LLM narrative summaries

In [ ]:
summaries = json.loads((DATA / "llm_summaries.json").read_text(encoding="utf-8"))
for place, groups in summaries.items():
    display(Markdown(f"### {PLACE_EN.get(place, place)}\n"
                     f"**KO reviewers:** {groups.get('ko', '')}\n\n"
                     f"**EN reviewers:** {groups.get('en', '')}"))

## 11. Geographic view

In [ ]:
fig("11_geo_bubble_seoul.png", 750)
fig("11_geo_bubble_busan.png", 750)